In [ ]:
from __future__ import annotations

import argparse
import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

import sys
sys.argv = ['']


TRAIN_FEATURE_PATHS = [
    Path("Extracted_Features/BoW_int.npy"),
    Path("Extracted_Features/Normalized_CH.npy"),
    Path("Extracted_Features/Normalized_CM55.npy"),
    Path("Extracted_Features/Normalized_CORR.npy"),
    Path("Extracted_Features/Normalized_EDH.npy"),
    Path("Extracted_Features/Normalized_WT.npy"),
]

TEST_FEATURE_PATHS = [
    Path("Extracted_Features_Test/BoW_int.npy"),
    Path("Extracted_Features_Test/Normalized_CH.npy"),
    Path("Extracted_Features_Test/Normalized_CM55.npy"),
    Path("Extracted_Features_Test/Normalized_CORR.npy"),
    Path("Extracted_Features_Test/Normalized_EDH.npy"),
    Path("Extracted_Features_Test/Normalized_WT.npy"),
]


@dataclass
class TrainConfig:
    batch_size: int = 128
    epochs: int = 50
    lr: float = 1e-3
    weight_decay: float = 1e-4
    hidden_dims: tuple[int, ...] = (1024, 512, 256)
    dropout: float = 0.3
    activation: str = "gelu"
    val_ratio: float = 0.1
    threshold: float = 0.5
    random_seed: int = 42
    projection_dim: int = 256


class FeatureGroupDataset(Dataset):

    def __init__(
        self,
        feature_groups: list[np.ndarray],
        labels: np.ndarray
    ):

        self.feature_groups = [

            torch.tensor(
                features,
                dtype=torch.float32
            )

            for features in feature_groups
        ]

        self.labels = torch.tensor(
            labels,
            dtype=torch.float32
        )

    def __len__(self) -> int:

        return len(self.labels)

    def __getitem__(
        self,
        idx: int
    ):

        return (

            [
                features[idx]
                for features
                in self.feature_groups
            ],

            self.labels[idx]
        )


class EarlyFusionMLP(nn.Module):

    def __init__(
        self,
        input_dims: list[int],
        projection_dim: int,
        hidden_dims: tuple[int, ...],
        output_dim: int,
        dropout: float,
        activation: str,
    ):
        super().__init__()

        self.projectors = nn.ModuleList([

            nn.Sequential(

                nn.Linear(
                    input_dim,
                    projection_dim
                ),

                make_activation(
                    activation
                )
            )

            for input_dim
            in input_dims
        ])

        layers: list[nn.Module] = []

        prev_dim = (
            projection_dim *
            len(input_dims)
        )

        for hidden_dim in hidden_dims:

            layers.append(
                nn.Linear(
                    prev_dim,
                    hidden_dim
                )
            )

            layers.append(
                make_activation(
                    activation
                )
            )

            if dropout > 0:

                layers.append(
                    nn.Dropout(
                        dropout
                    )
                )

            prev_dim = hidden_dim

        layers.append(
            nn.Linear(
                prev_dim,
                output_dim
            )
        )

        self.mlp = nn.Sequential(
            *layers
        )

    def forward(
        self,
        feature_groups: list[torch.Tensor]
    ) -> torch.Tensor:

        projected_groups = [

            projector(feature)

            for projector, feature
            in zip(
                self.projectors,
                feature_groups
            )
        ]

        fused = torch.cat(
            projected_groups,
            dim=1
        )

        return self.mlp(fused)


def set_seed(seed: int) -> None:

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:

    if torch.backends.mps.is_available():
        return torch.device("mps")

    if torch.cuda.is_available():
        return torch.device("cuda")

    return torch.device("cpu")


def load_array(path: Path) -> np.ndarray:

    if not path.exists():

        raise FileNotFoundError(
            f"Missing file: {path.resolve()}"
        )

    return np.load(path).astype(np.float32)


def load_feature_groups(
    data_root: Path,
    paths: list[Path]
) -> list[np.ndarray]:

    arrays = [

        load_array(data_root / path)

        for path in paths
    ]

    n_rows = {
        array.shape[0]
        for array in arrays
    }

    if len(n_rows) != 1:

        raise ValueError(
            f"Feature row counts do not match: "
            f"{sorted(n_rows)}"
        )

    return arrays


def make_activation(name: str) -> nn.Module:

    activations = {
        "relu": nn.ReLU,
        "gelu": nn.GELU,
        "silu": nn.SiLU,
    }

    if name not in activations:

        raise ValueError(
            f"Unsupported activation: {name}"
        )

    return activations[name]()


def parse_hidden_dims(
    value: str
) -> tuple[int, ...]:

    dims = tuple(
        int(part.strip())
        for part in value.split(",")
        if part.strip()
    )

    if not dims:

        raise argparse.ArgumentTypeError(
            "hidden dims must contain "
            "at least one integer"
        )

    return dims


def evaluate(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    threshold: float,
) -> dict[str, float]:

    model.eval()

    all_targets = []
    all_probs = []
    all_preds = []

    with torch.no_grad():

        for feature_groups, y in loader:

            feature_groups = [

                feature.to(device)

                for feature
                in feature_groups
            ]

            probs = torch.sigmoid(
                model(feature_groups)
            )

            all_probs.append(
                probs.cpu().numpy()
            )

            all_preds.append(
                (
                    probs > threshold
                ).float().cpu().numpy()
            )

            all_targets.append(
                y.numpy()
            )

    targets = np.vstack(all_targets)

    probs = np.vstack(all_probs)

    preds = np.vstack(all_preds)

    try:

        map_score = average_precision_score(
            targets,
            probs,
            average="macro"
        )

    except ValueError:

        map_score = 0.0

    return {

        "mAP": float(map_score),

        "micro_f1": float(
            f1_score(
                targets,
                preds,
                average="micro",
                zero_division=0
            )
        ),

        "macro_f1": float(
            f1_score(
                targets,
                preds,
                average="macro",
                zero_division=0
            )
        ),
    }


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> float:

    model.train()

    total_loss = 0.0

    total_seen = 0

    for feature_groups, y in loader:

        feature_groups = [

            feature.to(device)

            for feature
            in feature_groups
        ]

        y = y.to(device)

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(
            feature_groups
        )

        loss = criterion(
            logits,
            y
        )

        loss.backward()

        optimizer.step()

        total_loss += (
            loss.item() *
            len(y)
        )

        total_seen += len(y)

    return total_loss / max(total_seen, 1)


def save_json(
    path: Path,
    data: object
) -> None:

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    path.write_text(
        json.dumps(data, indent=2),
        encoding="utf-8"
    )


def main() -> None:

    parser = argparse.ArgumentParser(
        description=(
            "Train Early Fusion MLP baseline."
        )
    )

    parser.add_argument(
        "--data-root",
        type=Path,
        default=Path(
            "D:/Desktop/760_dataset"
        )
    )

    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path(
            "D:/Desktop/760_dataset/"
            "Early_Fusion_BCE/runs"
        )
    )

    parser.add_argument(
        "--epochs",
        type=int,
        default=50
    )

    parser.add_argument(
        "--batch-size",
        type=int,
        default=128
    )

    parser.add_argument(
        "--lr",
        type=float,
        default=1e-3
    )

    parser.add_argument(
        "--weight-decay",
        type=float,
        default=1e-4
    )

    parser.add_argument(
        "--threshold",
        type=float,
        default=0.5
    )

    parser.add_argument(
        "--val-ratio",
        type=float,
        default=0.1
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=42
    )

    parser.add_argument(
        "--num-workers",
        type=int,
        default=0
    )

    args = parser.parse_args()

    set_seed(args.seed)

    device = get_device()

    print(f"Using device: {device}")

    args.output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    print("\nLoading feature groups...")

    matched_indices = np.load(
        "matched_indices_no_overlap.npy"
    )

    print(
        f"Matched subset size: "
        f"{len(matched_indices)}"
    )

    train_groups = load_feature_groups(
        args.data_root,
        TRAIN_FEATURE_PATHS
    )

    train_groups = [

        group[matched_indices]

        for group
        in train_groups
    ]

    y_all = load_array(
        args.data_root /
        "database_labels_81_big.npy"
    )

    y_all = y_all[matched_indices]

    test_groups = load_feature_groups(
        args.data_root,
        TEST_FEATURE_PATHS
    )

    y_test = load_array(
        args.data_root /
        "database_labels_81_test.npy"
    )

    idx_train, idx_val = train_test_split(
        np.arange(len(y_all)),
        test_size=args.val_ratio,
        random_state=args.seed,
        shuffle=True,
    )

    train_loader = DataLoader(

        FeatureGroupDataset(

            [
                group[idx_train]
                for group
                in train_groups
            ],

            y_all[idx_train]
        ),

        batch_size=args.batch_size,

        shuffle=True,

        num_workers=args.num_workers,

        pin_memory=device.type == "cuda",
    )

    val_loader = DataLoader(

        FeatureGroupDataset(

            [
                group[idx_val]
                for group
                in train_groups
            ],

            y_all[idx_val]
        ),

        batch_size=args.batch_size,

        shuffle=False,

        num_workers=args.num_workers,

        pin_memory=device.type == "cuda",
    )

    test_loader = DataLoader(

        FeatureGroupDataset(
            test_groups,
            y_test
        ),

        batch_size=args.batch_size,

        shuffle=False,

        num_workers=args.num_workers,

        pin_memory=device.type == "cuda",
    )

    config = TrainConfig(

        batch_size=args.batch_size,

        epochs=args.epochs,

        lr=args.lr,

        weight_decay=args.weight_decay,

        hidden_dims=(1024, 512, 256),

        dropout=0.3,

        activation="gelu",

        val_ratio=args.val_ratio,

        threshold=args.threshold,

        random_seed=args.seed,

        projection_dim=256,
    )

    input_dims = [
        group.shape[1]
        for group
        in train_groups
    ]

    model = EarlyFusionMLP(

        input_dims=input_dims,

        projection_dim=config.projection_dim,

        hidden_dims=config.hidden_dims,

        output_dim=y_all.shape[1],

        dropout=config.dropout,

        activation=config.activation,

    ).to(device)

    optimizer = torch.optim.AdamW(

        model.parameters(),

        lr=config.lr,

        weight_decay=config.weight_decay,
    )

    criterion = nn.BCEWithLogitsLoss()

    best_map = -1.0

    history = []

    checkpoint_path = (
        args.output_dir /
        "best.pt"
    )

    print("\n" + "=" * 70)

    print("Training: Early Fusion")

    print("=" * 70)

    for epoch in range(
        1,
        config.epochs + 1
    ):

        train_loss = train_one_epoch(

            model,

            train_loader,

            optimizer,

            criterion,

            device,
        )

        val_metrics = evaluate(

            model,

            val_loader,

            device,

            config.threshold,
        )

        row = {

            "epoch": epoch,

            "train_loss": float(
                train_loss
            ),

            **{
                f"val_{k}": v
                for k, v
                in val_metrics.items()
            },
        }

        history.append(row)

        if val_metrics["mAP"] > best_map:

            best_map = val_metrics["mAP"]

            torch.save(

                {

                    "model_state_dict":
                    model.state_dict(),

                    "config":
                    asdict(config),

                    "num_classes":
                    y_all.shape[1],

                    "best_val_metrics":
                    val_metrics,
                },

                checkpoint_path,
            )

        print(

            f"Epoch "
            f"{epoch:03d}/"
            f"{config.epochs} "

            f"loss="
            f"{train_loss:.4f} "

            f"val_mAP="
            f"{val_metrics['mAP']:.4f} "

            f"val_micro_f1="
            f"{val_metrics['micro_f1']:.4f} "

            f"val_macro_f1="
            f"{val_metrics['macro_f1']:.4f}"
        )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    test_metrics = evaluate(

        model,

        test_loader,

        device,

        config.threshold,
    )

    save_json(
        args.output_dir /
        "training_history.json",
        history
    )

    save_json(
        args.output_dir /
        "test_metrics.json",
        test_metrics
    )

    save_json(
        args.output_dir /
        "config.json",
        asdict(config)
    )

    all_results = [

        {
            "Method": "Early Fusion",
            "mAP": test_metrics["mAP"],
            "Micro-F1": test_metrics["micro_f1"],
            "Macro-F1": test_metrics["macro_f1"],
        }
    ]

    save_json(

        args.output_dir /
        "all_results.json",

        all_results
    )

    print("\n" + "=" * 70)

    print("FINAL RESULT")

    print("=" * 70)

    print(
        f"Early Fusion -> "
        f"mAP={test_metrics['mAP']:.4f}, "
        f"Micro-F1={test_metrics['micro_f1']:.4f}, "
        f"Macro-F1={test_metrics['macro_f1']:.4f}"
    )


if __name__ == "__main__":

    main()

Using device: cuda

Loading feature groups...
Matched subset size: 116127

Training: Early Fusion


c:\Users\Yorushika\miniconda3\envs\yolov8\lib\site-packages\sklearn\metrics\_ranking.py:980: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Epoch 001/50 loss=0.0819 val_mAP=0.2123 val_micro_f1=0.5520 val_macro_f1=0.1142


c:\Users\Yorushika\miniconda3\envs\yolov8\lib\site-packages\sklearn\metrics\_ranking.py:980: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Epoch 002/50 loss=0.0668 val_mAP=0.2475 val_micro_f1=0.5567 val_macro_f1=0.1398


...

c:\Users\Yorushika\miniconda3\envs\yolov8\lib\site-packages\sklearn\metrics\_ranking.py:980: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Epoch 049/50 loss=0.0331 val_mAP=0.2693 val_micro_f1=0.5851 val_macro_f1=0.2430


c:\Users\Yorushika\miniconda3\envs\yolov8\lib\site-packages\sklearn\metrics\_ranking.py:980: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Epoch 050/50 loss=0.0328 val_mAP=0.2677 val_micro_f1=0.5876 val_macro_f1=0.2417

FINAL RESULT
Early Fusion -> mAP=0.3432, Micro-F1=0.6061, Macro-F1=0.2385


C:\Users\Yorushika\AppData\Local\Temp\ipykernel_22160\3119901423.py:803: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(
c:\Users\Yorushika\miniconda3